# Natya Posture Alignment - Static Hastas Training

Run this notebook in Google Colab to train the model on STATIC HASTAS (hand gestures).


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!pip install mediapipe opencv-python-headless pandas scikit-learn seaborn matplotlib tqdm


In [ ]:
# 2. Imports and Configuration
import os, glob, pickle, warnings, re
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from collections import defaultdict
warnings.filterwarnings('ignore')

# ----------------- CONFIGURATION -----------------
DRIVE_ROOT = '/content/drive/MyDrive/TrainingData'
IMAGES_DIR = f'{DRIVE_ROOT}/FinalHastas'
CSV_PATH = f'{DRIVE_ROOT}/Instructions/static_hastas_template.csv'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/Checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MIN_IMAGES = 2

FEATURES_CACHE = f'{CHECKPOINT_DIR}/hastas_features.npz'
RAW_CACHE = f'{CHECKPOINT_DIR}/hastas_raw_samples.pkl'
CKPT_PATH = f'{CHECKPOINT_DIR}/hastas_model.pt'

print(f'Device: {DEVICE}')
print(f'Reading images from: {IMAGES_DIR}')
print(f'Reading CSV from: {CSV_PATH}')


In [ ]:
# 3. MediaPipe Hand Utilities
# Feature Dim for Hands: 21 landmarks * 2 coords = 42
FEATURE_DIM = 42

# MediaPipe Initialization
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
mp_hands = vision.HandLandmarker.create_from_options(options)
print('MediaPipe Hand model ready.')

def pad_to_square(image: np.ndarray) -> np.ndarray:
    h, w = image.shape[:2]
    if h == w: return image
    size = max(h, w)
    pad_h = (size - h) // 2
    pad_w = (size - w) // 2
    return cv2.copyMakeBorder(image, pad_h, size - h - pad_h, pad_w, size - w - pad_w, cv2.BORDER_CONSTANT, value=[0, 0, 0])

def normalise_hand_landmarks(arr):
    # arr is (21, 3)
    arr = arr.copy()
    wrist = arr[0, :2]
    middle_mcp = arr[9, :2]
    scale = np.linalg.norm(middle_mcp - wrist)
    scale = max(scale, 1e-6)
    arr[:, :2] = (arr[:, :2] - wrist) / scale
    return arr

def extract_hand_from_image(image_path):
    frame = cv2.imread(image_path)
    if frame is None: return None
    
    frame = pad_to_square(frame)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = mp_hands.detect(mp_image)
    
    if not result.hand_landmarks: return None
    
    lm = result.hand_landmarks[0]
    arr = np.array([[l.x, l.y, l.z] for l in lm])
    
    seq_norm = normalise_hand_landmarks(arr)
    return seq_norm

def build_feature_vector(seq_norm):
    return seq_norm[:, :2].flatten()

def flip_sequence(seq_norm):
    flipped = seq_norm.copy()
    flipped[:, 0] *= -1
    return flipped

def add_noise(seq_norm, sigma=0.01):
    noisy = seq_norm.copy()
    noisy[:, :2] += np.random.normal(0, sigma, noisy[:, :2].shape)
    return noisy

def augment_sample(seq_norm):
    variants = []
    flipped = flip_sequence(seq_norm)
    variants.append((build_feature_vector(flipped), flipped))

    noisy = add_noise(seq_norm)
    variants.append((build_feature_vector(noisy), noisy))
    return variants


In [ ]:
# 4. Read Labels and Map to Images in Google Drive
print(f"Loading instructions from {CSV_PATH}")
df = pd.read_csv(CSV_PATH)

# Using columns for IDs and Labels
if 'Mudra_Name' in df.columns:
    label_col = 'Mudra_Name'
else:
    label_col = df.columns[1]

id_col = df.columns[0]
df[label_col] = df[label_col].astype(str).str.strip()
df[id_col] = df[id_col].astype(str).str.strip()

print(f"Loaded {len(df)} rows. Using '{id_col}' as ID and '{label_col}' as Label.")

all_drive_images = glob.glob(f"{IMAGES_DIR}/*")
print(f"Found {len(all_drive_images)} items in directory.")

image_files = {}
for _, row in df.iterrows():
    step_id = row[id_col]
    label = row[label_col]
    
    matched_file = None
    for vf in all_drive_images:
        if step_id in os.path.basename(vf):
            matched_file = vf
            break
    if matched_file: image_files[step_id] = {'path': matched_file, 'label': label}

print(f"Successfully matched {len(image_files)} images.")

class_counts = defaultdict(int)
for v in image_files.values(): class_counts[v['label']] += 1

valid_classes = {cls for cls, cnt in class_counts.items() if cnt >= MIN_IMAGES}
processing_list = [v for v in image_files.values() if v['label'] in valid_classes]
print(f"Total images to process: {len(processing_list)}")


In [ ]:
# 5. Extract Features and Cache Data
cache_valid = False
if os.path.exists(FEATURES_CACHE):
    try:
        data = np.load(FEATURES_CACHE, allow_pickle=True)
        if data['X'].shape[1] == FEATURE_DIM:
            X = data['X']; y = data['y']
            label_names = list(data['label_names'])
            cache_valid = True
            print("Loaded from cache!")
    except Exception as e: pass

raw_samples = []
if os.path.exists(RAW_CACHE):
    try:
        with open(RAW_CACHE, 'rb') as f: raw_samples = pickle.load(f)
    except: pass

if not cache_valid:
    failed = []
    if len(raw_samples) == 0:
        print('Extracting raw hand sequences...')
        for item in tqdm(processing_list):
            img_path = item['path']; cls = item['label']
            try:
                seq_norm = extract_hand_from_image(img_path)
                if seq_norm is None:
                    failed.append(img_path); continue
                fv = build_feature_vector(seq_norm)
                raw_samples.append((fv, cls, seq_norm, img_path))
            except Exception as e: failed.append(img_path)

        with open(RAW_CACHE, 'wb') as f: pickle.dump(raw_samples, f)
            
    temp_y = [item[1] for item in raw_samples]
    unique_labels = sorted(set(temp_y))
    class_sample_counts = {cls: int(np.sum(np.array(temp_y) == cls)) for cls in unique_labels}
    max_count = max(class_sample_counts.values()) if class_sample_counts else 0
    print(f'Balancing classes to max_count: {max_count}')
    
    X_aug, y_aug = [], []
    class_raw = defaultdict(list)
    for item in raw_samples: class_raw[item[1]].append(item)

    for cls in unique_labels:
        items = class_raw[cls]
        for fv, _, seq_norm, img_path in items:
            X_aug.append(fv); y_aug.append(cls)

        needed = max_count - len(items)
        if needed <= 0: continue

        pool = items.copy()
        added = 0
        while added < needed:
            src = pool[added % len(pool)]
            fv, _, seq_norm, img_path = src
            variants = augment_sample(seq_norm)
            for v_fv, v_seq in variants:
                if added >= needed: break
                X_aug.append(v_fv); y_aug.append(cls)
                added += 1

    X = np.array(X_aug); y = np.array(y_aug)
    label_names = sorted(set(y))
    np.savez(FEATURES_CACHE, X=X, y=y, label_names=label_names)
    print(f'Saved cache → {FEATURES_CACHE}. Samples: {len(X)}')


In [ ]:
# 6. Model Definition and Training Loop
le = LabelEncoder()
y_enc = le.fit_transform(y)
NUM_CLASSES = len(le.classes_)

X_mean = X.mean(axis=0)
X_std  = X.std(axis=0) + 1e-8
X_norm = (X - X_mean) / X_std

try:
    X_train, X_val, y_train, y_val = train_test_split(X_norm, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
except ValueError:
    X_train, X_val, y_train, y_val = train_test_split(X_norm, y_enc, test_size=0.2, random_state=42)

class HandClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, num_classes),
        )
    def forward(self, x): return self.net(x)

class HandDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

model = HandClassifier(FEATURE_DIM, NUM_CLASSES).to(DEVICE)
EPOCHS, BATCH, LR, WD = 100, 16, 1e-3, 1e-4

train_loader = DataLoader(HandDataset(X_train, y_train), batch_size=BATCH, shuffle=True, drop_last=True if len(X_train) > BATCH else False)
val_loader   = DataLoader(HandDataset(X_val, y_val), batch_size=BATCH, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

counts_tr = np.array([np.sum(y_train == i) for i in range(NUM_CLASSES)], dtype=float)
cw = 1.0 / (counts_tr + 1e-8); cw = cw / cw.sum() * NUM_CLASSES
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(cw).to(DEVICE))

best_val_acc, best_state = 0.0, None

for epoch in range(1, EPOCHS + 1):
    model.train()
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(Xb.to(DEVICE)), yb.to(DEVICE))
        loss.backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            correct += (model(Xb.to(DEVICE)).argmax(1) == yb.to(DEVICE)).sum().item()
    acc = correct / len(X_val) if len(X_val) > 0 else 0
    if acc > best_val_acc:
        best_val_acc = acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if epoch % 20 == 0: print(f'Epoch {epoch:3d}/{EPOCHS} | val_acc={acc:.2%}')

if best_state: model.load_state_dict(best_state)
print(f'Best val accuracy: {best_val_acc:.2%}')

ckpt = {
    'model_state':  best_state, 'label_encoder': le, 'X_mean': X_mean, 'X_std': X_std,
    'num_classes':  NUM_CLASSES, 'feature_dim':  FEATURE_DIM,
}
torch.save(ckpt, CKPT_PATH)
print(f'Checkpoint saved -> {CKPT_PATH}')

